In [1]:
# 1. 필요한 라이브러리를 불러온다.
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
from peft import get_peft_model, LoraConfig, TaskType

d:\dev\workspace\ai\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. 학습에 필요한 데이터를 불러온다.
df = pd.read_csv("./data/review_data.csv", encoding='cp949')

In [3]:
df.head(3)

,text,labels
0,배우들 연기도 너무 좋았어요.,1
1,스토리가 탄탄하고 연출도 훌륭했어요.,1
2,정말 감동적인 영화였습니다. 눈물이 멈추질 않았어요.,1


In [4]:
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['labels'], 
    random_state=0
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [5]:
# 3. LoRA를 적용할 기본 모델을 불러온다.
model_id = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_id, 
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 517.14it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

In [ ]:
# BertEncoder의 BertSelfAttention 내부에 Query, Key, Value에 대해 LoRa 적용이 가능하다.
print(base_model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(300, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [7]:
# 4. 데이터를 토크나이징을 통해 전처리한다.
def preprocess(data):
    return tokenizer(
        data["text"], 
        padding = "max_length",
        truncation = True, 
        max_length = 64)

train_dataset = train_dataset.map(
    preprocess, 
    batched = True,
    remove_columns = ["text", "__index_level_0__"]
)

test_dataset = test_dataset.map(
    preprocess, 
    batched=True,
    remove_columns=["text", "__index_level_0__"]
)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map: 100%|██████████| 20/20 [00:00<00:00, 196.23 examples/s]


In [11]:
# 5. LoraConfig는 LoRA를 어떻게 사용할지 구체적으로 설정하기 위해 사용하는 객체이다.
# 5-4행: SEQ_CLS는 Sequence Classification의 줄임말이며 시퀀스 분류라는 뜻을 가진다.
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"] 
)

lora_model = get_peft_model(base_model, peft_config)

In [ ]:
# 전체 파라미터 개수는 약 1억 1천만 개, 역전파로 학습되는 파라미터 개수는 약 30만 개. 
print(lora_model.print_trainable_parameters())

trainable params: 296,450 || all params: 109,216,516 || trainable%: 0.2714
None


In [13]:
# 파라미터를 저장할 때도 LoRA를 사용하면 30만 개만 저장하면 된다.
training_args = TrainingArguments(
    output_dir="./saved_models/lora_sentiment",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,

    logging_strategy="epoch",

    use_cpu=True
)

In [14]:
# 6. 학습기 설정을 한다.
def compute_metrics(predict):
    preds = np.argmax(predict.predictions, axis=1)
    acc = np.mean(preds == predict.label_ids)
    return {"accuracy": acc}

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [17]:
# 7. 학습을 진행한다.
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.677551,0.668294,0.650000
2,0.660517,0.665664,0.650000
3,0.676385,0.663692,0.650000
4,0.664544,0.661646,0.650000
5,0.647863,0.661076,0.650000


TrainOutput(global_step=50, training_loss=0.6653718662261963, metrics={'train_runtime': 425.0504, 'train_samples_per_second': 0.941, 'train_steps_per_second': 0.118, 'total_flos': 13201087488000.0, 'train_loss': 0.6653718662261963, 'epoch': 5.0})

In [ ]:
# 테스트를 위해 새로운 문장을 준비하고 토크나이징한다.
# 테스트 예측
test_texts = [
    # 긍정 데이터
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.",
    "스토리는 평범했지만 연출 덕분에 재미있었어요.",
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.",
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.",
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.",

    # 부정 데이터
    "이야기가 늘어져서 중간부터 집중이 안 됐어요.",
    "연출이 과해서 오히려 몰입을 방해했어요.",
    "캐릭터 행동이 이해되지 않아서 답답했어요.",
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.",
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요."
]


inputs = tokenizer(
    test_texts, 
    return_tensors="pt", 
    padding=True, 
    truncation=True, 
    max_length=64
)

In [ ]:
# 8. 학습된 모델을 활용해 예측
lora_model.eval()
with torch.no_grad():
    outputs = lora_model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)
print("예측 결과:", preds.tolist())

예측 결과: [0, 0, 1, 1, 1, 0, 0, 1, 1, 1]


In [ ]:
# 정확도는 50%가 나와 Full fine-tuning 보다 성능이 떨어지는 것을 볼 수 있다.
labels_target = torch.tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
accuracy = (preds == labels_target).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 0.5


In [22]:
# 학습된 모델과 토크나이저를 저장한다.
trainer.save_model("./saved_models/lora_sentiment/final_model")
tokenizer.save_pretrained("./saved_models/lora_sentiment/final_model")

('./saved_models/lora_sentiment/final_model\\tokenizer_config.json',
 './saved_models/lora_sentiment/final_model\\tokenizer.json')